# LEGO Data Quality

The [lego-casus](https://github.com/wortell-smart-learning/lego-casus) repository contains a dataset of 12 CSV files about LEGO parts, sets, inventories, colors, and themes. When loading this data, data type pitfalls can cause subtle bugs. In this notebook we'll explore common issues with numeric-looking string columns and how to handle them properly.

In [ ]:
import pandas as pd

## 1. Loading data with default dtypes

When you load a CSV file with `pd.read_csv()`, pandas automatically infers the data type of each column. Columns that contain only digits will be loaded as integers, even if the values are actually identifiers or codes that should remain strings.

This automatic inference is convenient most of the time, but it can cause problems when:
- Values have **leading zeros** (e.g. `"004511"` becomes `4511`)
- You need to do **string operations** on the column (e.g. pattern matching)
- The column contains a **mix** of numeric and non-numeric values

Let's see this in action with the LEGO parts dataset.

### Exercise 1

Load `parts.csv` using `pd.read_csv()`. Display the first 5 rows and the dtypes of the resulting DataFrame. Pay close attention to the dtype of the `part_num` column.

**Explanation:** We use `pd.read_csv()` with the default settings to load the file. Calling `.head()` shows the first 5 rows so we can visually inspect the data. The `.dtypes` attribute reveals the inferred data type for each column. Notice that `part_num` is loaded as `object` (string) because the column contains a mix of purely numeric and alphanumeric values, which prevents pandas from converting it to an integer.

In [ ]:
df = pd.read_csv("parts.csv")
print(df.head())
print()
print(df.dtypes)

### Exercise 2

Query the DataFrame for `part_num` equal to `"4511"` and `"004511"` separately using `df.query()`. These are two different LEGO parts — one is a sliding door, the other is a sticker sheet.

Note: this works because `part_num` happens to be loaded as a string (`object` dtype) since the column contains non-numeric values too. If it were loaded as an integer, `"004511"` would have lost its leading zeros and become indistinguishable from `"4511"`.

**Explanation:** We use `df.query()` with string comparisons to look up each part number. Since `part_num` is stored as a string, `"4511"` and `"004511"` are treated as different values. The first is a "Door Sliding - Type 1" and the second is a sticker sheet. If the column had been loaded as an integer, both would have been stored as `4511` and would be indistinguishable.

In [ ]:
print("Part 4511:")
print(df.query("part_num == '4511'"))
print()
print("Part 004511:")
print(df.query("part_num == '004511'"))

### Exercise 3

Use `df.query("part_num.str.contains('4511')")` to find all parts that contain `"4511"` somewhere in their part number. How many results do you get? This kind of string operation is only possible when the column is stored as a string.

**Explanation:** The `.str.contains()` method performs a substring search on each value in the column. This is a string-only operation — it would not work if `part_num` were numeric. We get 8 results because `"4511"` appears as a substring in various part numbers like `"004511"`, `"24511"`, `"44511"`, `"45117c01"`, and others. The `len()` call confirms the exact count.

In [ ]:
results = df.query("part_num.str.contains('4511')")
print(results)
print(f"\nNumber of results: {len(results)}")

## 2. The danger of numeric conversion

Sometimes you might be tempted to convert a column to numeric for sorting, calculations, or joins. However, for identifier columns this can be destructive:

- **Leading zeros are lost**: `"004511"` becomes `4511`, which is the same as `"4511"`
- **Non-numeric values become NaN**: part numbers like `"3626bpx19"` cannot be converted to a number
- **Distinct parts become duplicates**: two different parts may end up with the same numeric value

Let's see how much data we would lose or corrupt by converting `part_num` to numeric.

### Exercise 4

Create a copy of the DataFrame. Convert the `part_num` column to numeric using `pd.to_numeric(errors='coerce')` (this turns non-convertible values into `NaN`). Then:

1. Check how many rows have a `NaN` value for `part_num` after conversion (these are non-numeric part numbers that were lost).
2. Filter the converted DataFrame for rows where `part_num == 4511`. You should see that both the original `"4511"` and `"004511"` now have the same value.

**Explanation:** We use `.copy()` to avoid modifying the original DataFrame. `pd.to_numeric()` with `errors='coerce'` converts values that cannot be parsed as numbers into `NaN` instead of raising an error. Counting the NaN values with `.isna().sum()` shows how many part numbers are non-numeric and would be lost entirely. Filtering for `4511` shows that both the original `"4511"` and `"004511"` now map to the same numeric value, making them appear as duplicates.

In [ ]:
df_numeric = df.copy()
df_numeric["part_num"] = pd.to_numeric(df_numeric["part_num"], errors="coerce")

nan_count = df_numeric["part_num"].isna().sum()
print(f"Number of rows with NaN part_num after conversion: {nan_count}")
print()

print("Rows where part_num == 4511 (after numeric conversion):")
print(df_numeric.query("part_num == 4511"))

### Exercise 5

Count how many unique part numbers exist in the original DataFrame (as strings) versus in the numeric-converted copy (excluding NaN values). The difference tells you how many parts would be incorrectly merged by the conversion.

**Explanation:** We use `.nunique()` on the original string column to count all unique part numbers. For the numeric column, we first drop NaN values with `.dropna()` before counting unique values, because NaN values are not meaningful identifiers. The difference between these two counts has two components: part numbers that became NaN (completely lost) and part numbers that collapsed into duplicates due to leading zero removal. Both represent data corruption caused by the numeric conversion.

In [ ]:
unique_as_string = df["part_num"].nunique()
unique_as_numeric = df_numeric["part_num"].dropna().nunique()

print(f"Unique part numbers as strings:  {unique_as_string}")
print(f"Unique part numbers as numeric:  {unique_as_numeric}")
print(f"Difference (parts lost/merged):  {unique_as_string - unique_as_numeric}")

## 3. Specifying dtypes when loading

The best way to prevent these issues is to **explicitly specify the dtype** when loading the CSV file. The `pd.read_csv()` function accepts a `dtype` parameter where you can pass a dictionary mapping column names to their desired types.

```python
df = pd.read_csv("file.csv", dtype={"column_name": str})
```

This ensures that the column is always loaded as a string, regardless of its contents. This is especially important for:
- Part numbers, product codes, ZIP codes, phone numbers
- Any identifier that may have leading zeros
- Columns that you will use for string operations

See the [pandas documentation on `read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) for all available parameters.

### Exercise 6

Reload `parts.csv` with the `dtype` parameter set so that `part_num` is explicitly loaded as a string. Verify that `part_num` has dtype `object` (which is how pandas represents strings). Then show that `"004511"` is preserved correctly by querying for it.

**Explanation:** By passing `dtype={"part_num": str}` to `pd.read_csv()`, we override the automatic type inference for the `part_num` column and force it to be loaded as a string. We then verify this by checking the dtype (which should be `object`) and by querying for `"004511"` to confirm that the leading zeros are preserved. This approach is defensive and ensures correctness regardless of the column's contents.

In [ ]:
df_safe = pd.read_csv("parts.csv", dtype={"part_num": str})

print(f"dtype of part_num: {df_safe['part_num'].dtype}")
print()
print("Query for '004511':")
print(df_safe.query("part_num == '004511'"))

### Exercise 7

Write a general function `load_csv_safe(filepath, string_columns)` that:
- Takes a file path and a list of column names that should be forced to string type
- Builds the appropriate `dtype` dictionary
- Returns the loaded DataFrame

Test it by loading `parts.csv` with `part_num` as a string column. Verify the dtype of `part_num` in the result.

**Explanation:** We create a reusable function that builds a `dtype` dictionary from a list of column names using a dictionary comprehension. This is a practical pattern for projects where multiple CSV files need the same treatment — you can maintain a single list of identifier columns and pass it to the function every time you load data. The function wraps `pd.read_csv()` so it is easy to extend with additional parameters later (e.g. encoding, separator).

In [ ]:
def load_csv_safe(filepath, string_columns):
    dtype_dict = {col: str for col in string_columns}
    return pd.read_csv(filepath, dtype=dtype_dict)


df_result = load_csv_safe("parts.csv", ["part_num"])
print(f"dtype of part_num: {df_result['part_num'].dtype}")
print()
print(df_result.head())

## Summary

When loading CSV data with pandas, always be mindful of automatic dtype inference:

- **Check dtypes after loading** with `df.dtypes` — don't assume columns are the type you expect.
- **Numeric-looking strings** (part numbers, ZIP codes, product codes) can lose leading zeros or become indistinguishable when loaded as numbers.
- **Use the `dtype` parameter** in `pd.read_csv()` to explicitly force string types for identifier columns.
- **Be careful with `pd.to_numeric()`** — it can silently destroy data by coercing non-numeric values to `NaN` and removing leading zeros.

A good practice is to write a wrapper function (like `load_csv_safe`) that enforces string types for known identifier columns across your project.